# Baseline and IDF-Weighted Metadata Recommender
----------------
**MSc Data Science Dissertation**  
**Author:** Vikrant Deshmukh  
**University:** University of Bristol  
**Project:** Video Game Recommendation System

This notebook develops two recommendation approaches using the cleaned Steam Games Dataset 2025:

1. A non-personalised popularity baseline.
2. A query driven, metadata based recommender using genres, tags and gameplay modes.

The popularity baseline provides a transparent catalogue-level benchmark. The metadata recommender identifies structurally similar games using separately encoded and IDF-weighted metadata feature groups.

------

## Notebook Objectives

Notebook aims to:

- construct a non-personalised popularity baseline;
- build an IDF-weighted metadata recommender using genres, tags and play modes;
- generate explainable Top-N recommendations; and
- prepare metadata candidates for later hybrid fusion.

In [79]:
# Core libraries

import ast
import html
import json
import os
import re
import unicodedata

import pandas as pd
import numpy as np

from sklearn.neighbors import NearestNeighbors
from pathlib import Path

In [80]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Dataset Loading and Configuration

The cleaned Steam Games Dataset 2025 is loaded from Google Drive. Data cleaning, text preprocessing and the construction of derived fields were completed in the preceding data-preparation notebook.

This notebook therefore focuses only on recommendation modelling and does not repeat the complete cleaning pipeline.

In [81]:
# Configure the project and dataset paths.

if IN_COLAB:
    PROJECT_ROOT = Path("/content/drive/MyDrive/MSC_DISSERTATION")
else:
    PROJECT_ROOT = Path.cwd()

DATA_DIR = PROJECT_ROOT / "Data"
OUTPUT_DIR = PROJECT_ROOT / "Output/metadata_outputs"

data_path = DATA_DIR / "games_clean_v1.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found at: {data_path}")

games = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Dataset shape:", games.shape)

Dataset loaded successfully.
Dataset shape: (89618, 50)


In [82]:
games = pd.read_csv(data_path)

## 2. Dataset Overview and Validation

The dataset dimensions, sample records and available columns are inspected to confirm that the cleaned file has loaded correctly.

The recommendation approaches in this notebook primarily use:

- appid and display_name for game identification;
- genres, categories and tags for content-based similarity;
- recommendations, positive, negative and peak_ccu for the popularity baseline; and
- release and engagement fields for interpreting the resulting recommendations.

In [83]:
print("Dataset Shape: ", games.shape)
print("\n available columns:")
print(games.columns.tolist())

games.head(3)


Dataset Shape:  (89618, 50)

 available columns:
['appid', 'name', 'header_image', 'release_date', 'required_age', 'price', 'detailed_description', 'about_the_game', 'short_description', 'windows', 'mac', 'linux', 'metacritic_score', 'achievements', 'recommendations', 'developers', 'publishers', 'categories', 'genres', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'median_playtime_forever', 'peak_ccu', 'tags', 'pct_pos_total', 'num_reviews_total', 'pct_pos_recent', 'num_reviews_recent', 'release_year', 'display_name', 'name_clean', 'short_description_clean', 'about_the_game_clean', 'detailed_description_clean', 'genres_clean', 'categories_clean', 'tags_clean', 'developers_clean', 'publishers_clean', 'metadata_text', 'combined_text', 'log_recommendations', 'log_positive', 'log_negative', 'log_average_playtime_forever', 'log_median_playtime_forever', 'log_peak_ccu', 'log_num_reviews_total']


,appid,name,header_image,release_date,required_age,price,detailed_description,about_the_game,short_description,windows,...,publishers_clean,metadata_text,combined_text,log_recommendations,log_positive,log_negative,log_average_playtime_forever,log_median_playtime_forever,log_peak_ccu,log_num_reviews_total
0,730,Counter-Strike 2,https://shared.akamai.steamstatic.com/store_it...,2012-08-21,0,0.0,"For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...","For over two decades, Counter-Strike has offer...",True,...,valve,action free to play multi player cross platfor...,counter strike 2 action free to play multi pla...,15.297473,15.827852,13.942239,10.410004,8.551595,14.008077,15.971096
1,578080,PUBG: BATTLEGROUNDS,https://shared.akamai.steamstatic.com/store_it...,2017-12-21,0,0.0,"LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...","LAND, LOOT, SURVIVE! Play PUBG: BATTLEGROUNDS ...",Play PUBG: BATTLEGROUNDS for free. Land on str...,True,...,krafton inc,action adventure massively multiplayer free to...,pubg battlegrounds action adventure massively ...,14.364792,14.212917,13.839654,0.000000,0.000000,13.332201,14.737323
2,570,Dota 2,https://shared.akamai.steamstatic.com/store_it...,2013-07-09,0,0.0,"The most-played game on Steam. Every day, mill...","The most-played game on Steam. Every day, mill...","Every day, millions of players worldwide enter...",True,...,valve,action strategy free to play multi player co o...,dota 2 action strategy free to play multi play...,9.570669,14.507889,13.019974,10.669699,6.801283,13.228484,14.712658


### 2.1 Relevant Feature Inspection

The key fields required by the popularity and metadata recommendation pipelines are checked for availability and missing values before model construction.

In [84]:
required_cols = ["appid",
    "display_name",
    "metadata_text",
    "combined_text",
    "genres",
    "categories",
    "tags"
]

missing_required_cols = [
    col for col in required_cols
    if col not in games.columns
]

print("Missing required Columns: ", missing_required_cols)

Missing required Columns:  []


In [85]:
# null check

games[required_cols].isnull().sum()

,0
appid,0
display_name,0
metadata_text,0
combined_text,0
genres,0
categories,0
tags,0


## 3. Game Title Search Helper

A title-search helper is created to locate games in the catalogue using partial text matching.

This is useful because Steam titles may contain punctuation, trademark symbols, subtitles or edition names. The helper allows the exact dataset title and application identifier to be confirmed before generating recommendations.

In [86]:
def normalise_game_title(title):
  """Normalises Game titles by removing unnecessary symbols, tags and gaps"""
  title = unicodedata.normalize("NFKD", str(title))
  title = title.casefold()

  title = (
        title
        .replace("’", "'")
        .replace("‘", "'")
        .replace("`", "'")
        .replace("®", "")
        .replace("™", "")
    )

  title = re.sub(r"[^a-z0-9]+", " ", title)
  title = re.sub(r"\s+", " ", title).strip()

  return title

# Part A: Popularity Baseline

## 4. Popularity Signals

The popularity baseline uses the following catalogue-level engagement signals:

- number of Steam recommendations;
- number of positive reviews;
- number of negative reviews; and
- peak concurrent users.

This model is non-personalised and produces the same global ranking regardless of the query. It is therefore used as a simple catalogue-level benchmark against which the query-driven recommendation models can be compared.


In [87]:
popularity_cols = [
    "recommendations",
    "positive",
    "negative",
    "peak_ccu"
]

for col in popularity_cols:
    print(col, ":", col in games.columns)

recommendations : True
positive : True
negative : True
peak_ccu : True


In [88]:
for col in popularity_cols:
  print(col,  ":", games[col].isnull().sum())

recommendations : 0
positive : 0
negative : 0
peak_ccu : 0


## 5. Numeric Conversion and Missing-Value Handling

The popularity-related columns are converted to numeric values before calculating the baseline score.

Invalid or missing values are converted to zero. This prevents missing engagement information from causing calculation errors while retaining games that have incomplete popularity data.

In [89]:
for col in popularity_cols:
    games[col] = pd.to_numeric(games[col], errors="coerce").fillna(0)

In [90]:
games[popularity_cols].dtypes

,0
recommendations,int64
positive,int64
negative,int64
peak_ccu,int64


## 6. Popularity Score Construction and Sensitivity Check

Steam engagement variables are highly skewed because a small number of games accumulate very large numbers of recommendations, reviews, and concurrent players. **Logarithmic transformation** is therefore applied to reduce the influence of extreme values.

To construct a transparent non-personalised popularity baseline, two candidate scoring formulations are considered.

### Formulation 1: Engagement and Review-Based Popularity

The first formulation combines recommendations, positive reviews, negative reviews, and peak concurrent users:

$$
P_{1} =
\log(1+\text{recommendations})
+\log(1+\text{positive})
-\log(1+\text{negative})
+\log(1+\text{peak_ccu})
$$

This formulation attempts to capture both overall engagement and review sentiment. Positive engagement and player activity increase the score, while negative reviews reduce it.

However, directly subtracting the number of negative reviews can be difficult to interpret as a measure of popularity. Highly exposed games may accumulate large numbers of both positive and negative reviews simply because they have very large player populations.

### Formulation 2: Engagement-Based Popularity

The second formulation uses recommendations and peak concurrent users:

$$
P_{2} =
\log(1+\text{recommendations})
+\log(1+\text{peak_ccu})
$$

This formulation focuses more directly on catalogue-level engagement and player activity without incorporating review sentiment into the popularity score.

The two formulations are compared as a sensitivity check to examine how the treatment of review information affects the resulting rankings.

Since no user-level relevance ground truth is available for this baseline, the comparison is not intended to identify an empirically optimal popularity function. Instead, it is used to evaluate the stability and interpretability of the resulting rankings and to support the selection of a suitable baseline formulation.


In [91]:
# Popularity Score first approach (baseline)
games["popularity_score_ap1"] = (
    np.log1p(games["recommendations"]) +
    np.log1p(games["positive"]) -
    np.log1p(games["negative"]) +
    np.log1p(games["peak_ccu"])
)

In [92]:
# Popularity Score second approach (engagement)
games["popularity_score_ap2"] = (
    np.log1p(games["recommendations"]) +
    np.log1p(games["peak_ccu"])
)

In [93]:
# Compare the Top-20 games produced by both popularity formulations

top20_ap1 = (
    games[
        ["appid", "display_name", "popularity_score_ap1"]
    ]
    .sort_values(
        "popularity_score_ap1",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top20_ap1.insert(
    0,
    "rank_ap1",
    np.arange(1, len(top20_ap1) + 1)
)


top20_ap2 = (
    games[
        ["appid", "display_name", "popularity_score_ap2"]
    ]
    .sort_values(
        "popularity_score_ap2",
        ascending=False
    )
    .head(20)
    .reset_index(drop=True)
)

top20_ap2.insert(
    0,
    "rank_ap2",
    np.arange(1, len(top20_ap2) + 1)
)

In [94]:
# Compare rank positions across the two formulations

popularity_comparison = (
    top20_ap1[
        [
            "appid",
            "display_name",
            "rank_ap1"
        ]
    ]
    .merge(
        top20_ap2[
            [
                "appid",
                "rank_ap2"
            ]
        ],
        on="appid",
        how="outer"
    )
    .sort_values(
        ["rank_ap1", "rank_ap2"],
        na_position="last"
    )
    .reset_index(drop=True)
)

popularity_comparison

,appid,display_name,rank_ap1,rank_ap2
0,730,Counter-Strike 2,1.0,1.0
1,431960,Wallpaper Engine,2.0,6.0
2,413150,Stardew Valley,3.0,8.0
3,578080,PUBG: BATTLEGROUNDS,4.0,2.0
4,271590,Grand Theft Auto V Legacy,5.0,3.0
5,252490,Rust,6.0,4.0
6,105600,Terraria,7.0,11.0
7,227300,Euro Truck Simulator 2,8.0,15.0
8,4000,Garry's Mod,9.0,12.0
9,1086940,Baldur's Gate 3,10.0,13.0


In [95]:
ap1_games = set(top20_ap1["appid"])
ap2_games = set(top20_ap2["appid"])

common_games = ap1_games & ap2_games
union_games = ap1_games | ap2_games

jaccard_similarity = (
    len(common_games) / len(union_games)
)

print("Common games in Top-20:", len(common_games))
print(
    "Top-20 Jaccard similarity:",
    round(jaccard_similarity, 4)
)

Common games in Top-20: 14
Top-20 Jaccard similarity: 0.5385


## 7. Popularity Baseline Interpretation

The popularity baseline is a manually constructed heuristic rather than a learned ranking function. It is intended to provide a transparent, non-personalised catalogue-level benchmark for comparison with the query-driven recommendation models developed later in the system.

The selected formulation can be interpreted as a **quality-adjusted popularity score**. Recommendations and peak concurrent users represent broad measures of engagement and visibility, while positive and negative reviews introduce a reception component.

Negative reviews are incorporated as a penalty because high engagement alone does not necessarily indicate favourable reception. A game may attract a very large player base while also receiving substantial negative feedback. Reducing the score according to negative review volume therefore prevents highly exposed but poorly received titles from dominating the ranking solely because of their scale.

A sensitivity comparison was conducted against a simpler engagement-only formulation based on recommendations and peak concurrent users. Fourteen of the Top-20 games were shared between the two rankings, corresponding to a Jaccard similarity of approximately 0.54. This indicates that the negative-review component has a meaningful effect on ranking order rather than acting as a negligible adjustment.

The baseline is therefore used as:

- a transparent non-personalised benchmark;
- a catalogue-level indicator of both engagement and reception;
- a reference point for comparing the behaviour of query-driven recommenders; and
- a simple global ranking against which more targeted recommendation approaches can be discussed.

The approach nevertheless has important limitations. It favours established games with large player communities, does not account for individual preferences, and may under-represent niche titles with limited exposure. In addition, the scoring weights are heuristic rather than learned from user interaction data, so the resulting ranking should be interpreted as a practical baseline rather than an empirically optimal popularity model.

In [96]:
def recommend_popular(top_n=100):
    """Return the globally highest-ranked games from the popularity baseline."""

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    result = (
        games[
            [
                "appid",
                "display_name",
                "popularity_score_ap1",
                "recommendations",
                "positive",
                "negative",
                "peak_ccu",
            ]
        ]
        .sort_values(
            ["popularity_score_ap1", "recommendations"],
            ascending=[False, False],
        )
        .head(top_n)
        .reset_index(drop=True)
        .rename(columns={"display_name": "recommended_game"})
    )

    result.insert(
        0,
        "popularity_rank",
        np.arange(1, len(result) + 1),
    )

    return result


popularity_baseline_top20 = recommend_popular(top_n=10)
popularity_baseline_top20


,popularity_rank,appid,recommended_game,popularity_score_ap1,recommendations,positive,negative,peak_ccu
0,1,730,Counter-Strike 2,31.191162,4401572,7480813,1135108,1212356
1,2,431960,Wallpaper Engine,29.213506,809225,855816,17140,120461
2,3,413150,Stardew Valley,28.753825,729406,841413,13440,67309
3,4,578080,PUBG: BATTLEGROUNDS,28.070257,1732007,1487960,1024436,616738
4,5,271590,Grand Theft Auto V Legacy,28.009422,1803063,1719950,250012,117698
5,6,252490,Rust,27.943749,992825,1043708,152272,200902
6,7,105600,Terraria,27.899915,1098792,1344773,34460,30516
7,8,227300,Euro Truck Simulator 2,27.705225,669576,825685,22349,43539
8,9,4000,Garry's Mod,27.591147,984713,1106689,36727,32384
9,10,1086940,Baldur's Gate 3,27.578268,647659,709860,23273,48021


------------------

# Part B: IDF-Weighted Metadata Recommender

## 8. Metadata-Based Recommendation Approach

The metadata recommender identifies games with similar structural characteristics using:

- genres;
- Steam tags; and
- gameplay modes derived from Steam categories.

Rather than combining all metadata into a single text field, each metadata group is processed separately. This preserves the meaning of different feature types and allows field-specific weights to be applied.

Platform-support categories are retained for analysis but are excluded from the core similarity score because features such as Steam Cloud or Family Sharing provide limited information about gameplay similarity.

## 9. Parsing Structured Metadata

Genres and categories are stored as list-like strings, while tags are stored as dictionary-like values containing tag names and vote counts.

A parsing function converts these values into standard Python lists.

For tags, the metadata recommender uses the tag names as categorical features. Raw tag vote counts are not inserted as text tokens because numeric vote values do not represent independent semantic features.

The parsed values are stored in:

- genres_list;
- categories_list; and
- tags_list.

In [97]:
# confirming metadata fields
metadata_fields = [
    "genres",
    "categories",
    "tags",
]

In [98]:
import ast
import html
import re
import pandas as pd


def parse_metadata_field(value):
    """
    Convert metadata stored as lists, dictionaries, tuples,
    sets, or string representations into a clean Python list.
    """

    if pd.isna(value):
        return []

    # Already structured values
    if isinstance(value, dict):
        return list(value.keys())

    if isinstance(value, (list, tuple, set)):
        return list(value)

    value = str(value).strip()

    if not value:
        return []

    # Try parsing values such as:
    # "['Action', 'Adventure']"
    # "{'Action': 120, 'RPG': 90}"
    try:
        parsed_value = ast.literal_eval(value)

        if isinstance(parsed_value, dict):
            return list(parsed_value.keys())

        if isinstance(parsed_value, (list, tuple, set)):
            return list(parsed_value)

    except (ValueError, SyntaxError):
        pass

    # Fallback for comma, pipe, or semicolon-separated values
    return re.split(r"[,|;]", value)

In [99]:
def normalise_metadata_label(value):
    """
    Standardise metadata labels while preserving their meaning.
    """

    value = html.unescape(str(value))
    value = value.lower().strip()

    # Standardise common formatting differences
    value = value.replace("&", " and ")
    value = value.replace("-", " ")
    value = value.replace("_", " ")

    # Remove unnecessary punctuation
    value = re.sub(r"[^a-z0-9+ ]+", " ", value)

    # Remove repeated whitespace
    value = re.sub(r"\s+", " ", value).strip()

    return value

In [100]:
def clean_metadata_values(values):
    cleaned_values = set()

    for value in values:
        normalised_value = normalise_metadata_label(value)

        if normalised_value:
            cleaned_values.add(normalised_value)

    return sorted(cleaned_values)

In [101]:
for field in metadata_fields:
    games[f"{field}_list"] = games[field].apply(
        parse_metadata_field
    )

    games[f"{field}_list"] = games[f"{field}_list"].apply(
        clean_metadata_values
    )

In [102]:
games[
    [
        "display_name",
        "genres_list",
        "categories_list",
        "tags_list"
    ]
].head(10)

,display_name,genres_list,categories_list,tags_list
0,Counter-Strike 2,"[action, free to play]","[cross platform multiplayer, in app purchases,...","[action, co op, competitive, difficult, e spor..."
1,PUBG: BATTLEGROUNDS,"[action, adventure, free to play, massively mu...","[multi player, online pvp, pvp, remote play on...","[action, battle royale, co op, competitive, di..."
2,Dota 2,"[action, free to play, strategy]","[co op, in app purchases, multi player, steam ...","[action, action rpg, character customization, ..."
3,Grand Theft Auto V Legacy,"[action, adventure]","[co op, full controller support, multi player,...","[action, adventure, atmospheric, automobile si..."
4,Tom Clancy's Rainbow Six® Siege,[action],"[co op, full controller support, in app purcha...","[3d, action, co op, competitive, destruction, ..."
5,Team Fortress 2,"[action, free to play]","[captions available, commentary available, cro...","[action, cartoon, cartoony, class based, co op..."
6,Terraria,"[action, adventure, indie, rpg]","[co op, family sharing, full controller suppor...","[2d, action, adventure, atmospheric, building,..."
7,Rust,"[action, adventure, indie, massively multiplay...","[co op, cross platform multiplayer, in app pur...","[action, adventure, building, co op, crafting,..."
8,Garry's Mod,"[casual, indie, simulation]","[captions available, co op, cross platform mul...","[action, building, casual, co op, comedy, expl..."
9,Apex Legends™,"[action, adventure, free to play]","[co op, full controller support, in app purcha...","[action, battle royale, character customizatio..."


## 10. Metadata Feature Engineering

The parsed metadata is transformed into structured feature groups before fitting the recommendation model.

The main feature-engineering stages are:

1. canonicalising equivalent metadata labels;
2. separating gameplay modes from platform-support categories;
3. constructing independent feature matrices; and
4. combining the core matrices using field-specific weights.

In [103]:
metadata_parsing_summary = pd.DataFrame({
    "field": metadata_fields,

    "games_with_metadata": [
        games[f"{field}_list"].apply(len).gt(0).sum()
        for field in metadata_fields
    ],

    "games_without_metadata": [
        games[f"{field}_list"].apply(len).eq(0).sum()
        for field in metadata_fields
    ],

    "average_values_per_game": [
        games[f"{field}_list"].apply(len).mean()
        for field in metadata_fields
    ]
})

metadata_parsing_summary

,field,games_with_metadata,games_without_metadata,average_values_per_game
0,genres,89406,212,2.879154
1,categories,88742,876,4.196523
2,tags,72746,16872,11.258754


## 11. Metadata Coverage and Frequency Analysis
The metadata fields were successfully parsed into structured lists for genres, categories, and tags.

Genres and categories show high coverage across the dataset, with metadata available for more than 99% of games. Tags are more detailed and informative, but are missing for approximately 19% of games.

The frequency analysis also shows that some metadata values are extremely common. For example, indie, action, and adventure dominate the genre field, while single player, family sharing, and steam achievements dominate the category field.

This indicates that all metadata values should not contribute equally to similarity. Common labels may provide limited discriminative value, while more specific tags such as open world, psychological horror, story rich, and difficult can provide stronger signals of user preference.

Therefore, the improved recommender processes each metadata field separately and applies inverse document frequency weighting to reduce the influence of highly common labels.

In [104]:
def build_metadata_frequency(dataframe, list_column):
    """
    Create a frequency table showing how many games contain
    each metadata label.
    """

    exploded_values = (
        dataframe[list_column]
        .explode()
        .dropna()
    )

    frequency_table = (
        exploded_values
        .value_counts()
        .rename_axis("label")
        .reset_index(name="game_count")
    )

    frequency_table["game_percentage"] = (
        frequency_table["game_count"]
        / len(dataframe)
        * 100
    ).round(3)

    return frequency_table


genre_frequency = build_metadata_frequency(
    games,
    "genres_list"
)

category_frequency = build_metadata_frequency(
    games,
    "categories_list"
)

tag_frequency = build_metadata_frequency(
    games,
    "tags_list"
)

In [105]:
print("Unique genres:", len(genre_frequency))
print("Unique categories:", len(category_frequency))
print("Unique tags:", len(tag_frequency))

Unique genres: 33
Unique categories: 40
Unique tags: 452


## 12. Structured Metadata Feature Engineering

### 12.1 Metadata Label Canonicalisation

A small number of metadata labels describe the same feature using slightly different wording. These variants are standardised so that equivalent labels are represented by a single feature during similarity calculation.

For example, `vr support` and `vr supported` are merged into the canonical label `vr supported`.


In [106]:
CATEGORY_CANONICAL_MAP = {
    "vr support": "vr supported"
}


def canonicalise_metadata_values(values, mapping):
    """
    Replace known label variants with canonical labels
    and remove any duplicates created during the process.
    """

    canonical_values = {
        mapping.get(value, value)
        for value in values
        if value
    }

    return sorted(canonical_values)


games["categories_list"] = games["categories_list"].apply(
    lambda values: canonicalise_metadata_values(
        values,
        CATEGORY_CANONICAL_MAP
    )
)

In [107]:
category_frequency_updated = build_metadata_frequency(
    games,
    "categories_list"
)

category_frequency_updated[
    category_frequency_updated["label"].str.contains(
        "vr",
        case=False,
        na=False
    )
]

,label,game_count,game_percentage
16,vr only,5090,5.680
26,vr supported,1189,1.327
38,steamvr collectibles,38,0.042


### 12.2 Separating Gameplay and Platform Features

Steam categories contain both gameplay information and platform-support functionality.

Gameplay-related features include single-player, multiplayer, player-versus-player, cooperative play, online cooperative play, local or split-screen play, and massively multiplayer functionality.

Platform-support features include Steam Achievements, Steam Cloud, Family Sharing, controller support, Remote Play, and accessibility or interface support.

These groups are separated into:

- `play_mode_features`
- `platform_features`

Only play-mode features are included in the core recommendation representation.


In [108]:
PLAY_MODE_LABELS = {
    "single player",
    "multi player",
    "pvp",
    "online pvp",
    "lan pvp",
    "co op",
    "online co op",
    "lan co op",
    "mmo",
    "shared split screen",
    "shared split screen pvp",
    "shared split screen co op",
    "cross platform multiplayer",
    "remote play together"
}

In [109]:
games["play_mode_features"] = games["categories_list"].apply(
    lambda values: sorted([
        value
        for value in values
        if value in PLAY_MODE_LABELS
    ])
)

games["platform_features"] = games["categories_list"].apply(
    lambda values: sorted([
        value
        for value in values
        if value not in PLAY_MODE_LABELS
    ])
)



## 13. Building Separate IDF-Weighted Metadata Matrices

Genres, tags, play modes and platform features are encoded separately.

For each feature group:

1. MultiLabelBinarizer creates a sparse binary matrix indicating whether each game contains each label.
2. TfidfTransformer applies inverse document frequency weighting.
3. L2 normalisation prepares the vectors for cosine similarity.

In this context, term frequency represents binary feature presence. IDF weighting reduces the importance of labels shared by most games and increases the contribution of comparatively distinctive labels.

In [110]:
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfTransformer

### 13.1 Matrix-Building Function

A reusable matrix-building function is used to ensure that every metadata group follows the same encoding procedure.

For each feature group, the function returns:

the fitted multi-label encoder;
the fitted IDF transformer; and
the resulting sparse, normalised feature matrix.

This modular structure allows individual metadata groups to be examined, weighted and compared independently.

In [111]:
def build_idf_weighted_matrix(metadata_series):
    """
    Create a sparse IDF-weighted matrix from a column
    containing lists of categorical metadata labels.
    """

    encoder = MultiLabelBinarizer(sparse_output=True)

    # Binary presence/absence matrix
    binary_matrix = encoder.fit_transform(metadata_series)

    # Reduce the influence of very common labels
    idf_transformer = TfidfTransformer(
        norm="l2",
        use_idf=True,
        smooth_idf=True,
        sublinear_tf=False
    )

    weighted_matrix = idf_transformer.fit_transform(
        binary_matrix
    ).astype(np.float32)

    return encoder, idf_transformer, weighted_matrix

In [112]:
genre_encoder, genre_idf, genre_matrix = (
    build_idf_weighted_matrix(
        games["genres_list"]
    )
)

play_mode_encoder, play_mode_idf, play_mode_matrix = (
    build_idf_weighted_matrix(
        games["play_mode_features"]
    )
)

tag_encoder, tag_idf, tag_matrix = (
    build_idf_weighted_matrix(
        games["tags_list"]
    )
)

platform_encoder, platform_idf, platform_matrix = (
    build_idf_weighted_matrix(
        games["platform_features"]
    )
)

In [113]:
matrix_summary = pd.DataFrame({
    "feature_group": [
        "Genres",
        "Play-mode features",
        "Tags",
        "Platform features"
    ],

    "games": [
        genre_matrix.shape[0],
        play_mode_matrix.shape[0],
        tag_matrix.shape[0],
        platform_matrix.shape[0]
    ],

    "unique_features": [
        genre_matrix.shape[1],
        play_mode_matrix.shape[1],
        tag_matrix.shape[1],
        platform_matrix.shape[1]
    ],

    "non_zero_values": [
        genre_matrix.nnz,
        play_mode_matrix.nnz,
        tag_matrix.nnz,
        platform_matrix.nnz
    ],

    "matrix_density_percent": [
        round(
            genre_matrix.nnz
            / (genre_matrix.shape[0] * genre_matrix.shape[1])
            * 100,
            4
        ),

        round(
            play_mode_matrix.nnz
            / (play_mode_matrix.shape[0] * play_mode_matrix.shape[1])
            * 100,
            4
        ),

        round(
            tag_matrix.nnz
            / (tag_matrix.shape[0] * tag_matrix.shape[1])
            * 100,
            4
        ),

        round(
            platform_matrix.nnz
            / (platform_matrix.shape[0] * platform_matrix.shape[1])
            * 100,
            4
        )
    ]
})

matrix_summary

,feature_group,games,unique_features,non_zero_values,matrix_density_percent
0,Genres,89618,33,258024,8.7247
1,Play-mode features,89618,14,158183,12.6077
2,Tags,89618,452,1008987,2.4909
3,Platform features,89618,25,217827,9.7225


## 14. Constructing the Weighted Core Metadata Matrix

The core metadata representation combines three feature groups:

- **Tags: 60%**
- **Genres: 25%**
- **Play modes: 15%**

The field weights were defined as model-design assumptions during development rather than being learned from labelled relevance data.

Tags receive the largest weight because they provide the most fine-grained description of a game. Compared with broad genre labels, Steam tags can capture gameplay mechanics, themes, perspective, difficulty, atmosphere, and other characteristics that may distinguish games within the same genre.

Genres are assigned the second-largest weight because they provide an important high-level description of game type, but they are generally less specific than tags. Many games share broad genres such as Action, Adventure, or Indie, which limits their discriminative value when used alone.

Play-mode features receive a smaller complementary weight. These features describe how a game is played, for example single-player, multiplayer, cooperative, or player-versus-player modes. Although useful for distinguishing interaction styles, they provide less information about the thematic or mechanical similarity of games.

Platform-support features are excluded from the core recommendation matrix because features such as Steam Cloud, controller support, Family Sharing, and achievements provide limited information about gameplay similarity.

The weights therefore reflect the intended relative importance of the three metadata groups rather than an empirically optimal configuration. Their role is to prevent the larger and more detailed tag representation from being treated identically to broader genre and play-mode information.

Each independently normalised feature matrix is multiplied by the square root of its assigned field weight before concatenation. This ensures that, when cosine similarity is calculated on the combined representation, the contribution of each feature group corresponds approximately to its intended field-level importance.

The final combined matrix is then L2-normalised before nearest-neighbour retrieval.

In [114]:
from scipy.sparse import hstack
from sklearn.preprocessing import normalize


In [115]:
CORE_METADATA_WEIGHTS = {
    "tags": 0.60,
    "genres": 0.25,
    "play_modes": 0.15
}

assert np.isclose(
    sum(CORE_METADATA_WEIGHTS.values()),
    1.0
), "Core metadata weights must sum to 1."

In [116]:
weighted_tag_matrix = (
    tag_matrix
    * np.sqrt(CORE_METADATA_WEIGHTS["tags"])
)

weighted_genre_matrix = (
    genre_matrix
    * np.sqrt(CORE_METADATA_WEIGHTS["genres"])
)

weighted_play_mode_matrix = (
    play_mode_matrix
    * np.sqrt(CORE_METADATA_WEIGHTS["play_modes"])
)

In [117]:
core_metadata_matrix = hstack(
    [
        weighted_tag_matrix,
        weighted_genre_matrix,
        weighted_play_mode_matrix
    ],
    format="csr"
).astype(np.float32)

In [118]:
core_metadata_matrix = normalize(
    core_metadata_matrix,
    norm="l2",
    axis=1,
    copy=False
)

In [119]:
core_matrix_density = (
    core_metadata_matrix.nnz
    / (
        core_metadata_matrix.shape[0]
        * core_metadata_matrix.shape[1]
    )
    * 100
)

core_matrix_memory_mb = (
    core_metadata_matrix.data.nbytes
    + core_metadata_matrix.indices.nbytes
    + core_metadata_matrix.indptr.nbytes
) / (1024 ** 2)

print("Core metadata matrix shape:",
      core_metadata_matrix.shape)

print("Non-zero values:",
      core_metadata_matrix.nnz)

print("Matrix density:",
      round(core_matrix_density, 4), "%")

print("Approximate sparse memory usage:",
      round(core_matrix_memory_mb, 2), "MB")

Core metadata matrix shape: (89618, 499)
Non-zero values: 1425194
Matrix density: 3.187 %
Approximate sparse memory usage: 11.22 MB


In [120]:
core_row_norms = np.sqrt(
    core_metadata_matrix
    .multiply(core_metadata_matrix)
    .sum(axis=1)
).A1

print(
    "Games with no usable core metadata:",
    np.sum(core_row_norms == 0)
)

print(
    "Minimum non-zero row norm:",
    core_row_norms[core_row_norms > 0].min()
)

print(
    "Maximum row norm:",
    core_row_norms.max()
)

Games with no usable core metadata: 105
Minimum non-zero row norm: 0.9999999
Maximum row norm: 1.0000001


## 15. Fitting the Metadata Nearest-Neighbours Model

A brute-force nearest-neighbours model is fitted using cosine distance.

Cosine similarity is appropriate for the sparse, L2-normalised metadata representation because it measures the orientation of the feature vectors rather than their absolute magnitude.

The number of retrieved neighbours is controlled by the `candidate_pool` parameter of the recommendation function. For qualitative inspection, smaller Top-N recommendation lists are displayed, while a larger candidate pool is used when preparing recommendations for later hybrid fusion.

For hybrid integration, up to 100 nearest metadata candidates are retrieved and the highest-ranked 50 candidates are retained. This provides a sufficiently broad candidate set for rank fusion while keeping the subsequent hybrid stage computationally manageable.

In [121]:
metadata_eligible_mask = core_row_norms > 0

eligible_game_positions = np.flatnonzero(
    metadata_eligible_mask
)

eligible_core_metadata_matrix = (
    core_metadata_matrix[metadata_eligible_mask]
)

print(
    "Games included in Metadata V2:",
    eligible_core_metadata_matrix.shape[0]
)

print(
    "Games excluded due to missing core metadata:",
    (~metadata_eligible_mask).sum()
)

print(
    "Eligible matrix shape:",
    eligible_core_metadata_matrix.shape
)

Games included in Metadata V2: 89513
Games excluded due to missing core metadata: 105
Eligible matrix shape: (89513, 499)


In [122]:
# Model row → original games dataframe position
model_row_to_game_position = eligible_game_positions


# Original games dataframe position → model row
game_position_to_model_row = np.full(
    len(games),
    -1,
    dtype=np.int32
)

game_position_to_model_row[
    eligible_game_positions
] = np.arange(
    len(eligible_game_positions),
    dtype=np.int32
)

In [123]:
from sklearn.neighbors import NearestNeighbors

In [124]:
metadata_nn = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=51,
    n_jobs=-1
)

metadata_nn.fit(
    eligible_core_metadata_matrix
)

NearestNeighbors(algorithm='brute', metric='cosine', n_jobs=-1, n_neighbors=51)

In [125]:
print("Metadata V2 Nearest Neighbours model fitted successfully.")
print(
    "Indexed games:",
    metadata_nn.n_samples_fit_
)

Metadata V2 Nearest Neighbours model fitted successfully.
Indexed games: 89513


In [126]:
games["normalised_display_name"] = (
    games["display_name"]
    .apply(normalise_game_title)
)


In [127]:
title_to_game_position = {}

for game_position, title_key in enumerate(
    games["normalised_display_name"]
):
    if not title_key:
        continue

    if title_key not in title_to_game_position:
        title_to_game_position[title_key] = game_position

    else:
        existing_position = title_to_game_position[
            title_key
        ]

        existing_is_eligible = (
            game_position_to_model_row[
                existing_position
            ] >= 0
        )

        current_is_eligible = (
            game_position_to_model_row[
                game_position
            ] >= 0
        )

        if current_is_eligible and not existing_is_eligible:
            title_to_game_position[
                title_key
            ] = game_position

assert len(title_to_game_position) > 0, (
    "The title-to-game-position mapping was not created."
)

print(
    "Unique normalised titles mapped:",
    f"{len(title_to_game_position):,}"
)


Unique normalised titles mapped: 86,928


In [128]:
def search_game_titles(query, limit=10):
    """
    Return possible game-title matches.
    """

    query_key = normalise_game_title(query)

    matches = games[
        games["normalised_display_name"].str.contains(
            query_key,
            regex=False,
            na=False
        )
    ]

    return (
        matches["display_name"]
        .drop_duplicates()
        .head(limit)
        .tolist()
    )

In [129]:
search_game_titles(
    "elden ring"
)

['ELDEN RING']

In [130]:
def calculate_field_similarity(
    matrix,
    query_position,
    candidate_position
):
    """
    Calculate cosine similarity between two normalised
    sparse metadata rows.
    """

    return float(
        matrix[query_position]
        .multiply(matrix[candidate_position])
        .sum()
    )

## 16. Metadata Recommendation Function

The final recommendation function:

validates the requested game title;
identifies its catalogue and model positions;
retrieves the nearest candidate games;
removes the query game from its own results;
converts cosine distance into similarity;
calculates field-level similarity scores;
identifies shared metadata features; and
returns the ranked Top-N recommendations.

For later hybrid integration, the essential output fields are:

appid;
recommended_game;
metadata_score; and
metadata_rank.

The explainability columns are retained for model interpretation and dissertation analysis.

In [131]:
def recommend_metadata(
    game_title,
    top_n=10,
    candidate_pool=50
):
    """
    Generate game recommendations using the structured,
    field-weighted Metadata V2 representation.

    Parameters
    ----------
    game_title : str
        Exact or case-insensitive title of the query game.

    top_n : int, default=10
        Number of recommendations to return.

    candidate_pool : int, default=50
        Number of nearest metadata candidates to retrieve
        before returning the final Top-N results.

    Returns
    -------
    pandas.DataFrame
        Ranked metadata recommendations with overall and
        field-level similarity explanations.
    """

    if top_n < 1:
        raise ValueError("top_n must be at least 1.")

    if candidate_pool < top_n:
        candidate_pool = top_n

    title_key = normalise_game_title(game_title)

    # Check whether the requested title exists
    if title_key not in title_to_game_position:

        possible_matches = search_game_titles(
            game_title,
            limit=10
        )

        raise ValueError(
            f"Game title not found: {game_title}. "
            f"Possible matches: {possible_matches}"
        )

    query_position = title_to_game_position[
        title_key
    ]

    query_model_row = game_position_to_model_row[
        query_position
    ]

    # Query game has no usable core metadata
    if query_model_row == -1:
        raise ValueError(
            f"'{game_title}' does not contain sufficient "
            "genres, tags, or play-mode metadata."
        )

    number_of_neighbours = min(
        candidate_pool + 1,
        eligible_core_metadata_matrix.shape[0]
    )

    distances, neighbour_model_rows = (
        metadata_nn.kneighbors(
            eligible_core_metadata_matrix[
                query_model_row
            ],
            n_neighbors=number_of_neighbours
        )
    )

    recommendations = []

    query_genres = set(
        games.iloc[query_position]["genres_list"]
    )

    query_tags = set(
        games.iloc[query_position]["tags_list"]
    )

    query_play_modes = set(
        games.iloc[query_position][
            "play_mode_features"
        ]
    )

    for distance, neighbour_model_row in zip(
        distances[0],
        neighbour_model_rows[0]
    ):

        candidate_position = (
            model_row_to_game_position[
                neighbour_model_row
            ]
        )

        # Exclude the query game itself
        if candidate_position == query_position:
            continue

        candidate_row = games.iloc[
            candidate_position
        ]

        genre_similarity = calculate_field_similarity(
            genre_matrix,
            query_position,
            candidate_position
        )

        tag_similarity = calculate_field_similarity(
            tag_matrix,
            query_position,
            candidate_position
        )

        play_mode_similarity = calculate_field_similarity(
            play_mode_matrix,
            query_position,
            candidate_position
        )

        candidate_genres = set(
            candidate_row["genres_list"]
        )

        candidate_tags = set(
            candidate_row["tags_list"]
        )

        candidate_play_modes = set(
            candidate_row["play_mode_features"]
        )

        recommendation = {
            "appid":
                candidate_row["appid"],

            "recommended_game":
                candidate_row["display_name"],

            # Keep full precision for ranking.
            "metadata_score":
                1 - float(distance),

            "genre_similarity":
                round(genre_similarity, 4),

            "tag_similarity":
                round(tag_similarity, 4),

            "play_mode_similarity":
                round(play_mode_similarity, 4),

            "shared_genres":
                sorted(
                    query_genres &
                    candidate_genres
                ),

            "shared_tags":
                sorted(
                    query_tags &
                    candidate_tags
                ),

            "shared_play_modes":
                sorted(
                    query_play_modes &
                    candidate_play_modes
                )
        }

        # Add descriptive and engagement fields where available
        optional_columns = [
            "release_date",
            "pct_pos_total",
            "num_reviews_total",
            "recommendations"
        ]

        for column in optional_columns:
            if column in games.columns:
                recommendation[column] = (
                    candidate_row[column]
                )

        recommendations.append(
            recommendation
        )

        if len(recommendations) >= candidate_pool:
            break

    result = pd.DataFrame(
        recommendations
    )

    if result.empty:
        return result

    # Rank purely by metadata similarity.
    # AppID is used only as a deterministic tie-breaker.
    result = (
        result
        .sort_values(
            ["metadata_score", "appid"],
            ascending=[False, True],
            kind="mergesort"
        )
        .head(top_n)
        .reset_index(drop=True)
    )

    result.insert(
        0,
        "metadata_rank",
        np.arange(1, len(result) + 1)
    )

    # Round only after ranking so display remains readable
    # without affecting recommendation order.
    result["metadata_score"] = (
        result["metadata_score"].round(4)
    )

    return result

## 17. Development Case Studies

During the development of the metadata recommender, four query games were selected for qualitative inspection of recommendation behaviour:

- **ELDEN RING**, representing a Souls-like action RPG;
- **Red Dead Redemption 2**, representing an open-world narrative action game;
- **PUBG: BATTLEGROUNDS**, representing a multiplayer battle-royale game; and
- **Assassin's Creed Odyssey**, representing a franchise-oriented action-adventure game.

These games were deliberately chosen because they exhibit different combinations of genres, tags, gameplay modes, popularity levels, and franchise characteristics. This made it possible to inspect how the metadata representation behaved under different recommendation scenarios.

The case studies were used to examine several aspects of the recommender, including the retrieval of games with similar gameplay characteristics, the influence of shared tags and genres, franchise-related recommendations, the appearance of obscure or low-popularity candidates, and cases where highly similar metadata may lead to clone-like recommendations.

These four games were used only during model development and qualitative inspection. Their recommendation outputs were considered when understanding the behaviour and limitations of the metadata model, but they are not included in the final held-out evaluation set.

The final evaluation therefore uses a separate collection of query games that were not used during model development. This separation reduces the risk of evaluating the recommender using the same examples that influenced model-design decisions.


In [132]:
development_queries = [
    "ELDEN RING",
    "Red Dead Redemption 2",
    "PUBG: BATTLEGROUNDS",
    "Assassin's Creed Odyssey"
]

In [133]:
def generate_metadata_candidates(
    query_title,
    top_n=50,
    candidate_pool=100
):
    """
    Generate and standardise metadata candidates
    for hybrid recommendation.
    """

    results = recommend_metadata(
        game_title=query_title,
        top_n=top_n,
        candidate_pool=candidate_pool
    ).copy()

    title_key = normalise_game_title(query_title)

    query_position = title_to_game_position[
        title_key
    ]

    query_row = games.iloc[
        query_position
    ]

    results.insert(
        0,
        "query_appid",
        query_row["appid"]
    )

    results.insert(
        1,
        "query_game",
        query_row["display_name"]
    )

    return results

In [134]:
search_game_titles("elden ring")
search_game_titles("red dead redemption")
search_game_titles("pubg")
search_game_titles("odyssey")

["Assassin's Creed® Odyssey",
 'Ancestors: The Humankind Odyssey',
 'ENSLAVED™: Odyssey to the West™ Premium Edition',
 'ONE PIECE ODYSSEY',
 'Abyss Odyssey',
 "Orbo's Odyssey",
 '3030 Deathwar Redux - A Space Odyssey',
 'Jumplight Odyssey',
 'Satellite Odyssey: Prologue',
 'Etrian Odyssey HD']

In [135]:
metadata_candidate_tables = []

for game_title in development_queries:

    print(f"Generating candidates for: {game_title}")

    candidates = generate_metadata_candidates(
        query_title=game_title,
        top_n=50,
        candidate_pool=100
    )

    metadata_candidate_tables.append(
        candidates
    )

metadata_candidates = pd.concat(
    metadata_candidate_tables,
    ignore_index=True
)

metadata_candidates.shape

Generating candidates for: ELDEN RING
Generating candidates for: Red Dead Redemption 2
Generating candidates for: PUBG: BATTLEGROUNDS
Generating candidates for: Assassin's Creed Odyssey


(200, 16)

In [136]:
for game_title in development_queries:
    print("\n" + "=" * 80)
    print(f"Recommendations for: {game_title}")
    print("=" * 80)

    display(
        recommend_metadata(
            game_title=game_title,
            top_n=10,
            candidate_pool=50
        )
    )


Recommendations for: ELDEN RING


,metadata_rank,appid,recommended_game,metadata_score,genre_similarity,tag_similarity,play_mode_similarity,shared_genres,shared_tags,shared_play_modes,release_date,pct_pos_total,num_reviews_total,recommendations
0,1,678960,CODE VEIN,0.7459,1.0000,0.6319,0.7782,"[action, rpg]","[action, action rpg, character customization, ...","[co op, multi player, online co op, single pla...",2019-09-26,84.0,42021.0,42013
1,2,485510,Nioh: Complete Edition,0.7434,1.0000,0.5723,1.0000,"[action, rpg]","[action, action rpg, atmospheric, co op, dark ...","[co op, multi player, online co op, online pvp...",2017-11-07,78.0,26100.0,26058
2,3,335300,DARK SOULS™ II: Scholar of the First Sin,0.7291,1.0000,0.6517,0.5868,"[action, rpg]","[action, action rpg, atmospheric, character cu...","[co op, multi player, single player]",2015-04-01,84.0,72245.0,72139
3,4,1448440,Wo Long: Fallen Dynasty,0.7284,0.8633,0.6043,1.0000,"[action, rpg]","[action, action rpg, atmospheric, character cu...","[co op, multi player, online co op, online pvp...",2023-03-03,47.0,19736.0,19733
4,5,1358700,STRANGER OF PARADISE FINAL FANTASY ORIGIN,0.7246,0.8633,0.6535,0.7782,"[action, rpg]","[3d, action, action rpg, atmospheric, dark fan...","[co op, multi player, online co op, single pla...",2023-04-06,83.0,2779.0,2776
5,6,637650,FINAL FANTASY XV WINDOWS EDITION,0.7133,0.8196,0.5973,1.0000,[rpg],"[action, action rpg, atmospheric, co op, fanta...","[co op, multi player, online co op, online pvp...",2018-03-06,83.0,40759.0,40750
6,7,2130510,Divinus Vanitas,0.6983,0.8633,0.6334,0.6832,"[action, rpg]","[3d, action, action rpg, character customizati...","[co op, multi player, online co op, online pvp...",2022-09-14,74.0,27.0,0
7,8,649950,Ashen,0.6977,0.8633,0.6086,0.7782,"[action, rpg]","[action, action rpg, atmospheric, dark fantasy...","[co op, multi player, online co op, single pla...",2019-12-09,68.0,3005.0,3006
8,9,1325200,Nioh 2 – The Complete Edition,0.6918,1.0000,0.5418,0.7782,"[action, rpg]","[action, atmospheric, character customization,...","[co op, multi player, online co op, single pla...",2021-02-05,88.0,30120.0,30099
9,10,374320,DARK SOULS™ III,0.6828,0.5730,0.7526,0.5868,[action],"[action, action rpg, atmospheric, character cu...","[co op, multi player, single player]",2016-04-11,94.0,258335.0,258226



Recommendations for: Red Dead Redemption 2


,metadata_rank,appid,recommended_game,metadata_score,genre_similarity,tag_similarity,play_mode_similarity,shared_genres,shared_tags,shared_play_modes,release_date,pct_pos_total,num_reviews_total,recommendations
0,1,1404210,Red Dead Online,0.7675,1.0000,0.6150,0.9900,"[action, adventure]","[action, adventure, first person, gore, horses...","[co op, multi player, online co op, online pvp...",2020-12-01,83.0,56651.0,56631
1,2,271590,Grand Theft Auto V Legacy,0.6962,1.0000,0.4936,1.0000,"[action, adventure]","[action, adventure, atmospheric, first person,...","[co op, multi player, online co op, online pvp...",2015-04-13,87.0,1803832.0,1803063
2,3,2176790,Furry Arena [18+],0.6325,1.0000,0.0000,1.0000,"[action, adventure]",[],"[co op, multi player, online co op, online pvp...",2022-12-01,66.0,512.0,512
3,4,287700,METAL GEAR SOLID V: THE PHANTOM PAIN,0.6286,1.0000,0.5346,0.3858,"[action, adventure]","[action, adventure, atmospheric, great soundtr...","[multi player, single player]",2015-09-01,91.0,65758.0,65711
4,5,360430,Mafia III: Definitive Edition,0.6270,1.0000,0.5931,0.1410,"[action, adventure]","[action, adventure, atmospheric, fps, gore, gr...",[single player],2020-05-19,57.0,33735.0,33730
5,6,447040,Watch_Dogs® 2,0.6136,1.0000,0.4592,0.5868,"[action, adventure]","[action, adventure, atmospheric, fps, mature, ...","[co op, multi player, single player]",2016-11-28,82.0,74558.0,74360
6,7,235600,Tom Clancy’s Splinter Cell Blacklist,0.6111,1.0000,0.4551,0.5868,"[action, adventure]","[action, adventure, atmospheric, first person,...","[co op, multi player, single player]",2013-08-20,80.0,17277.0,17265
7,8,1938090,Call of Duty®,0.6096,0.6999,0.5108,0.8544,[action],"[action, atmospheric, first person, fps, gore,...","[co op, multi player, online co op, online pvp...",2022-10-27,55.0,379855.0,379597
8,9,904590,Arcade LA Deadzone,0.6079,1.0000,0.0000,0.8964,"[action, adventure]",[],"[co op, multi player, online co op, online pvp]",2019-02-27,NaN,0.0,0
9,10,312660,Sniper Elite 4,0.6076,1.0000,0.4493,0.5868,"[action, adventure]","[action, adventure, first person, fps, gore, m...","[co op, multi player, single player]",2017-02-13,91.0,50473.0,50468



Recommendations for: PUBG: BATTLEGROUNDS


,metadata_rank,appid,recommended_game,metadata_score,genre_similarity,tag_similarity,play_mode_similarity,shared_genres,shared_tags,shared_play_modes,release_date,pct_pos_total,num_reviews_total,recommendations
0,1,994200,天际起源 The Ark of Horizon,0.8334,1.0000,0.7223,1.0000,"[action, adventure, free to play, massively mu...","[action, battle royale, co op, competitive, ea...","[multi player, online pvp, pvp]",2019-03-13,52.0,671.0,0
1,2,779610,DEATH FIELD: The Battle Royale of Disaster,0.8100,0.7322,0.8414,0.8140,"[action, adventure, massively multiplayer]","[action, battle royale, co op, competitive, di...","[multi player, online pvp]",2018-05-03,26.0,152.0,152
2,3,433850,Z1 Battle Royale,0.7470,1.0000,0.5784,1.0000,"[action, adventure, free to play, massively mu...","[action, battle royale, co op, early access, f...","[multi player, online pvp, pvp]",2018-02-28,55.0,204340.0,144242
3,4,1000500,The Undisputables : Online Multiplayer Shooter,0.7178,0.7731,0.7196,0.6181,"[action, massively multiplayer]","[action, battle royale, co op, competitive, fi...","[multi player, online pvp, pvp]",2021-12-05,80.0,15.0,0
4,5,731250,ExoTanks,0.7121,0.9533,0.6422,0.5897,"[action, free to play, massively multiplayer]","[action, co op, competitive, early access, fir...","[multi player, online pvp, pvp]",2020-12-20,66.0,183.0,0
5,6,2262200,FUBG: FIGHT UNKNOWN BATTLEGROUND,0.7004,0.8298,0.6752,0.5853,"[action, adventure, massively multiplayer]","[action, battle royale, early access, first pe...","[multi player, pvp]",2024-05-25,NaN,0.0,0
6,7,439370,Midair,0.6997,0.9307,0.5284,1.0000,"[action, free to play, massively multiplayer]","[action, co op, competitive, early access, fir...","[multi player, online pvp, pvp]",2018-05-03,55.0,848.0,0
7,8,2645030,Chapter Wars,0.6956,0.7997,0.5808,0.9815,"[action, massively multiplayer]","[action, battle royale, co op, first person, f...","[multi player, online pvp, pvp]",2023-11-17,100.0,10.0,0
8,9,1121710,Total Lockdown,0.6829,0.8298,0.5425,1.0000,"[action, adventure, massively multiplayer]","[action, battle royale, competitive, fps, mult...","[multi player, online pvp, pvp]",2020-03-25,76.0,713.0,713
9,10,879160,Battlerite Royale,0.6710,0.9533,0.4757,0.9815,"[action, free to play, massively multiplayer]","[action, battle royale, competitive, early acc...","[multi player, online pvp, pvp]",2019-02-19,74.0,8202.0,3367



Recommendations for: Assassin's Creed Odyssey


,metadata_rank,appid,recommended_game,metadata_score,genre_similarity,tag_similarity,play_mode_similarity,shared_genres,shared_tags,shared_play_modes,release_date,pct_pos_total,num_reviews_total,recommendations
0,1,582160,Assassin's Creed® Origins,0.7866,1.0000,0.6444,1.0,"[action, adventure, rpg]","[action, action rpg, adventure, assassin, expl...",[single player],2017-10-26,85.0,102363.0,102229
1,2,3035570,Assassin's Creed Mirage,0.7804,0.7067,0.7562,1.0,"[action, adventure]","[action, action adventure, action rpg, adventu...",[single player],2024-10-17,74.0,5886.0,5767
2,3,2208920,Assassin's Creed Valhalla,0.7566,1.0000,0.5943,1.0,"[action, adventure, rpg]","[action, action adventure, action rpg, adventu...",[single player],2022-12-06,70.0,26946.0,26774
3,4,2221920,Immortals Fenyx Rising,0.7468,1.0000,0.5779,1.0,"[action, adventure, rpg]","[action, action adventure, action rpg, adventu...",[single player],2022-12-15,71.0,3380.0,3334
4,5,911400,Assassin's Creed® III Remastered,0.7377,0.7067,0.6850,1.0,"[action, adventure]","[action, action adventure, action rpg, adventu...",[single player],2019-03-29,61.0,10829.0,10772
5,6,260210,Assassin’s Creed® Liberation HD,0.7370,0.7067,0.6839,1.0,"[action, adventure]","[action, action adventure, adventure, assassin...",[single player],2014-01-15,52.0,2666.0,2659
6,7,368500,Assassin's Creed® Syndicate,0.7031,0.7067,0.6274,1.0,"[action, adventure]","[action, adventure, assassin, female protagoni...",[single player],2015-11-18,80.0,27248.0,27204
7,8,471010,Seven: Enhanced Edition,0.7019,0.9429,0.5269,1.0,"[action, adventure, rpg]","[action, action adventure, action rpg, adventu...",[single player],2017-12-01,76.0,1314.0,1309
8,9,714250,Eternity: The Last Unicorn,0.6987,0.9429,0.5217,1.0,"[action, adventure, rpg]","[action, action adventure, action rpg, adventu...",[single player],2019-03-05,39.0,104.0,104
9,10,354380,Assassin’s Creed® Chronicles: China,0.6790,0.7067,0.5872,1.0,"[action, adventure]","[action, adventure, assassin, female protagoni...",[single player],2015-04-21,73.0,3333.0,3330


## 18. Candidate Validation and Export

The exported candidate lists are validated before being passed to the hybrid
recommender. Each query should contain 50 unique candidate games, with no
missing application identifiers or duplicate query-candidate pairs.

In [137]:
query_counts = (
    metadata_candidates
    .groupby("query_game")
    .size()
)

duplicate_pairs = metadata_candidates.duplicated(
    subset=["query_appid", "appid"]
).sum()

missing_appids = metadata_candidates["appid"].isna().sum()

expected_rows = len(development_queries) * 50

print("Candidates per query:")
print(query_counts)
print("\nCombined candidate shape:", metadata_candidates.shape)
print("Duplicate query-candidate pairs:", duplicate_pairs)
print("Missing candidate appids:", missing_appids)

assert metadata_candidates.shape[0] == expected_rows, (
    f"Expected {expected_rows} rows, "
    f"but found {metadata_candidates.shape[0]}."
)
assert (query_counts == 50).all(), (
    "Every query game must have exactly 50 candidates."
)
assert duplicate_pairs == 0, (
    "Duplicate query-candidate pairs were found."
)
assert missing_appids == 0, (
    "Missing candidate appids were found."
)

print("\nMetadata candidate validation passed.")


Candidates per query:
query_game
Assassin's Creed® Odyssey    50
ELDEN RING                   50
PUBG: BATTLEGROUNDS          50
Red Dead Redemption 2        50
dtype: int64

Combined candidate shape: (200, 16)
Duplicate query-candidate pairs: 0
Missing candidate appids: 0

Metadata candidate validation passed.


In [138]:
output_directory = (
    "/content/drive/MyDrive/MSC_DISSERTATION/Output/metadata_outputs"
)

os.makedirs(
    output_directory,
    exist_ok=True,
)

metadata_output_path = os.path.join(
    output_directory,
    "metadata_candidates.csv",
)

popularity_output_path = os.path.join(
    output_directory,
    "popularity_baseline_top100.csv",
)

config_output_path = os.path.join(
    output_directory,
    "metadata_model_config.json",
)

metadata_candidates.to_csv(
    metadata_output_path,
    index=False,
)

popularity_baseline_top100 = recommend_popular(
    top_n=100,
)

popularity_baseline_top100.to_csv(
    popularity_output_path,
    index=False,
)

metadata_model_config = {
    "model_name": "IDF-weighted structured metadata recommender",
    "tag_weight": float(CORE_METADATA_WEIGHTS["tags"]),
    "genre_weight": float(CORE_METADATA_WEIGHTS["genres"]),
    "play_mode_weight": float(CORE_METADATA_WEIGHTS["play_modes"]),
    "distance_metric": "cosine",
    "nearest_neighbour_algorithm": "brute",
    "candidate_pool": 100,
    "exported_candidates_per_query": 50,
    "case_study_queries": development_queries,
}

with open(
    config_output_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        metadata_model_config,
        file,
        indent=4,
    )

print(f"Metadata candidates saved to:\n{metadata_output_path}")
print(f"Popularity baseline saved to:\n{popularity_output_path}")
print(f"Model configuration saved to:\n{config_output_path}")


Metadata candidates saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/metadata_outputs/metadata_candidates.csv
Popularity baseline saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/metadata_outputs/popularity_baseline_top100.csv
Model configuration saved to:
/content/drive/MyDrive/MSC_DISSERTATION/Output/metadata_outputs/metadata_model_config.json


In [139]:
saved_metadata_candidates = pd.read_csv(
    metadata_output_path
)

saved_popularity_baseline = pd.read_csv(
    popularity_output_path
)

print(
    "Saved metadata candidate shape:",
    saved_metadata_candidates.shape,
)

print(
    "Saved popularity baseline shape:",
    saved_popularity_baseline.shape,
)

display(saved_metadata_candidates.head(10))
display(saved_popularity_baseline.head(10))


Saved metadata candidate shape: (200, 16)
Saved popularity baseline shape: (100, 8)


,query_appid,query_game,metadata_rank,appid,recommended_game,metadata_score,genre_similarity,tag_similarity,play_mode_similarity,shared_genres,shared_tags,shared_play_modes,release_date,pct_pos_total,num_reviews_total,recommendations
0,1245620,ELDEN RING,1,678960,CODE VEIN,0.7459,1.0000,0.6319,0.7782,"['action', 'rpg']","['action', 'action rpg', 'character customizat...","['co op', 'multi player', 'online co op', 'sin...",2019-09-26,84.0,42021.0,42013
1,1245620,ELDEN RING,2,485510,Nioh: Complete Edition,0.7434,1.0000,0.5723,1.0000,"['action', 'rpg']","['action', 'action rpg', 'atmospheric', 'co op...","['co op', 'multi player', 'online co op', 'onl...",2017-11-07,78.0,26100.0,26058
2,1245620,ELDEN RING,3,335300,DARK SOULS™ II: Scholar of the First Sin,0.7291,1.0000,0.6517,0.5868,"['action', 'rpg']","['action', 'action rpg', 'atmospheric', 'chara...","['co op', 'multi player', 'single player']",2015-04-01,84.0,72245.0,72139
3,1245620,ELDEN RING,4,1448440,Wo Long: Fallen Dynasty,0.7284,0.8633,0.6043,1.0000,"['action', 'rpg']","['action', 'action rpg', 'atmospheric', 'chara...","['co op', 'multi player', 'online co op', 'onl...",2023-03-03,47.0,19736.0,19733
4,1245620,ELDEN RING,5,1358700,STRANGER OF PARADISE FINAL FANTASY ORIGIN,0.7246,0.8633,0.6535,0.7782,"['action', 'rpg']","['3d', 'action', 'action rpg', 'atmospheric', ...","['co op', 'multi player', 'online co op', 'sin...",2023-04-06,83.0,2779.0,2776
5,1245620,ELDEN RING,6,637650,FINAL FANTASY XV WINDOWS EDITION,0.7133,0.8196,0.5973,1.0000,['rpg'],"['action', 'action rpg', 'atmospheric', 'co op...","['co op', 'multi player', 'online co op', 'onl...",2018-03-06,83.0,40759.0,40750
6,1245620,ELDEN RING,7,2130510,Divinus Vanitas,0.6983,0.8633,0.6334,0.6832,"['action', 'rpg']","['3d', 'action', 'action rpg', 'character cust...","['co op', 'multi player', 'online co op', 'onl...",2022-09-14,74.0,27.0,0
7,1245620,ELDEN RING,8,649950,Ashen,0.6977,0.8633,0.6086,0.7782,"['action', 'rpg']","['action', 'action rpg', 'atmospheric', 'dark ...","['co op', 'multi player', 'online co op', 'sin...",2019-12-09,68.0,3005.0,3006
8,1245620,ELDEN RING,9,1325200,Nioh 2 – The Complete Edition,0.6918,1.0000,0.5418,0.7782,"['action', 'rpg']","['action', 'atmospheric', 'character customiza...","['co op', 'multi player', 'online co op', 'sin...",2021-02-05,88.0,30120.0,30099
9,1245620,ELDEN RING,10,374320,DARK SOULS™ III,0.6828,0.5730,0.7526,0.5868,['action'],"['action', 'action rpg', 'atmospheric', 'chara...","['co op', 'multi player', 'single player']",2016-04-11,94.0,258335.0,258226


,popularity_rank,appid,recommended_game,popularity_score_ap1,recommendations,positive,negative,peak_ccu
0,1,730,Counter-Strike 2,31.191162,4401572,7480813,1135108,1212356
1,2,431960,Wallpaper Engine,29.213506,809225,855816,17140,120461
2,3,413150,Stardew Valley,28.753825,729406,841413,13440,67309
3,4,578080,PUBG: BATTLEGROUNDS,28.070257,1732007,1487960,1024436,616738
4,5,271590,Grand Theft Auto V Legacy,28.009422,1803063,1719950,250012,117698
5,6,252490,Rust,27.943749,992825,1043708,152272,200902
6,7,105600,Terraria,27.899915,1098792,1344773,34460,30516
7,8,227300,Euro Truck Simulator 2,27.705225,669576,825685,22349,43539
8,9,4000,Garry's Mod,27.591147,984713,1106689,36727,32384
9,10,1086940,Baldur's Gate 3,27.578268,647659,709860,23273,48021


## 19. Conclusion

This notebook implemented two recommendation approaches using the cleaned Steam Games Dataset 2025.

First, a non-personalised quality-adjusted popularity baseline was developed using Steam recommendations, positive and negative reviews, and peak concurrent users. The baseline provides a transparent catalogue-level benchmark and intentionally incorporates negative reviews as a reception penalty so that highly exposed but poorly received games do not dominate the ranking solely because of their scale.

A sensitivity comparison with a simpler engagement-only formulation showed that the treatment of review information had a meaningful effect on the resulting ranking. This supports interpreting the selected formulation as a quality-adjusted popularity heuristic rather than a direct measure of player activity.

Second, an IDF-weighted structured metadata recommender was developed using genres, Steam tags, and gameplay modes. These feature groups were encoded separately, weighted using inverse document frequency, and combined using field-specific design weights. Cosine nearest-neighbour retrieval was then used to identify games with similar metadata profiles.

The metadata recommender produced interpretable recommendations, particularly for games with distinctive and complete metadata. Separate genre, tag, and gameplay-mode similarity scores also provide an explanation of the metadata characteristics shared between the query game and each recommended title.

The development case studies also highlighted limitations of metadata-only recommendation. Games with missing or sparse tags may rely heavily on broad genre and play-mode information, while obscure or clone-like titles can receive high similarity scores despite limited player engagement or weaker semantic relevance.

The metadata recommender therefore serves as one candidate-generation component of the later hybrid recommendation system. For hybrid integration, the highest-ranked 50 metadata candidates are retained and combined with candidates from the semantic and graph-derived recommenders using the Steam application identifier.

The four development query games used in this notebook are retained only for qualitative model inspection. Formal evaluation is conducted using a separate held-out set of query games that were not used during model development.

## Evaluation Candidate Generation

For formal model evaluation, the final metadata recommender is applied to ten representative query games covering different genres and gameplay styles.

The model generates 50 candidates per query so that the same candidate pool can later be used by the hybrid recommender. Only the Top 10 results will be used when calculating evaluation metrics.

In [140]:
evaluation_games = [
    "Counter-Strike 2",
    "Grand Theft Auto V Legacy",
    "Stardew Valley",
    "Sid Meier's Civilization VI",
    "Hollow Knight",
    "The Witcher 3: Wild Hunt",
    "Hades",
    "Deep Rock Galactic",
    "Factorio",
    "Phasmophobia"
]

print("Number of evaluation games:", len(evaluation_games))

Number of evaluation games: 10


In [141]:
# Check whether every evaluation game is available
# in the metadata recommender catalogue.

missing_evaluation_games = []

for game_title in evaluation_games:

    title_key = normalise_game_title(
        game_title
    )

    if title_key not in title_to_game_position:

        missing_evaluation_games.append(
            game_title
        )

if len(missing_evaluation_games) == 0:

    print(
        "All evaluation games are available."
    )

else:

    print(
        "Games not found:",
        missing_evaluation_games
    )

All evaluation games are available.


In [142]:
import time

metadata_evaluation_tables = []
metadata_runtime_records = []

for game_title in evaluation_games:

    print("Generating metadata candidates for:", game_title)

    start_time = time.perf_counter()

    candidates = generate_metadata_candidates(
        query_title=game_title,
        top_n=50,
        candidate_pool=100
    )

    end_time = time.perf_counter()

    query_runtime = end_time - start_time

    metadata_evaluation_tables.append(
        candidates
    )

    runtime_record = {
        "model": "Metadata",
        "query_game": game_title,
        "runtime_seconds": query_runtime
    }

    metadata_runtime_records.append(
        runtime_record
    )

metadata_evaluation_candidates = pd.concat(
    metadata_evaluation_tables,
    ignore_index=True
)

metadata_runtime_results = pd.DataFrame(
    metadata_runtime_records
)

print(
    "\nCandidate dataset shape:",
    metadata_evaluation_candidates.shape
)

print(
    "Runtime dataset shape:",
    metadata_runtime_results.shape
)

Generating metadata candidates for: Counter-Strike 2
Generating metadata candidates for: Grand Theft Auto V Legacy
Generating metadata candidates for: Stardew Valley
Generating metadata candidates for: Sid Meier's Civilization VI
Generating metadata candidates for: Hollow Knight
Generating metadata candidates for: The Witcher 3: Wild Hunt
Generating metadata candidates for: Hades
Generating metadata candidates for: Deep Rock Galactic
Generating metadata candidates for: Factorio
Generating metadata candidates for: Phasmophobia

Candidate dataset shape: (500, 16)
Runtime dataset shape: (10, 3)


### Evaluation Candidate Validation and Export

The generated metadata candidates are checked for completeness, duplicate recommendations and accidental inclusion of the query game.

The validated candidate results and query runtimes are then exported for use in the separate evaluation notebook.

In [143]:
# Check the number of query games.
query_count = metadata_evaluation_candidates[
    "query_game"
].nunique()

# Check the number of candidates for each query game.
candidate_counts = metadata_evaluation_candidates.groupby(
    "query_game"
).size()

# Check duplicate recommendations within each query.
duplicate_count = metadata_evaluation_candidates.duplicated(
    subset=[
        "query_appid",
        "appid"
    ]
).sum()

# Check whether a query game recommends itself.
self_recommendations = metadata_evaluation_candidates[
    metadata_evaluation_candidates["query_appid"]
    ==
    metadata_evaluation_candidates["appid"]
]

print("Number of query games:", query_count)
print("\nCandidates per query:")
print(candidate_counts)

print(
    "\nDuplicate recommendations:",
    duplicate_count
)

print(
    "Self-recommendations:",
    len(self_recommendations)
)

Number of query games: 10

Candidates per query:
query_game
Counter-Strike 2                50
Deep Rock Galactic              50
Factorio                        50
Grand Theft Auto V Legacy       50
Hades                           50
Hollow Knight                   50
Phasmophobia                    50
Sid Meier’s Civilization® VI    50
Stardew Valley                  50
The Witcher 3: Wild Hunt        50
dtype: int64

Duplicate recommendations: 0
Self-recommendations: 0


In [144]:
# Define output file paths.

metadata_evaluation_path = (
    "/content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/"
    "metadata_evaluation_candidates.csv"
)

metadata_runtime_path = (
    "/content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/"
    "metadata_evaluation_runtime.csv"
)

# Save evaluation candidates.
metadata_evaluation_candidates.to_csv(
    metadata_evaluation_path,
    index=False
)

# Save runtime results.
metadata_runtime_results.to_csv(
    metadata_runtime_path,
    index=False
)

print(
    "Candidate file saved to:",
    metadata_evaluation_path
)

print(
    "Runtime file saved to:",
    metadata_runtime_path
)

Candidate file saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/metadata_evaluation_candidates.csv
Runtime file saved to: /content/drive/MyDrive/MSC_DISSERTATION/Output/evaluation_outputs/metadata_evaluation_runtime.csv
